# EduTech Global · 03 · Attrition Visualisations

Build the visual evidence pack that supports Deliverable 2 (Workforce Stability Dashboard). Heatmaps and segmented breakdowns to identify intervention points.

## Setup — auto-resolving paths

Run this cell first.

In [ ]:


from pathlib import Path

def find_project_root():
    p = Path.cwd().resolve()
    for parent in [p] + list(p.parents):
        if parent.name == "edutech":
            return parent
        if (parent / "scripts").exists() and (parent / "outputs").exists() and (parent / "data").exists():
            return parent
    return Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROJECT_ROOT = find_project_root()
DATASET_DIR  = PROJECT_ROOT.parent / "dataset"
DATA_DIR     = PROJECT_ROOT / "data"
OUTPUTS_DIR  = PROJECT_ROOT / "outputs"
FIGURES_DIR  = PROJECT_ROOT / "figures"

for d in [DATA_DIR, OUTPUTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Dataset dir  : {DATASET_DIR}")
print(f"Data dir     : {DATA_DIR}")
print(f"Outputs dir  : {OUTPUTS_DIR}")
print(f"Figures dir  : {FIGURES_DIR}")

# Allow either dataset/ or data/ to hold the source CSV; accept both filenames
HR_CSV_NAMES = ["WA_Fn-UseC_-HR-Employee-Attrition.csv", "HR-Employee-Attrition.csv"]
hr_locations = []
for name in HR_CSV_NAMES:
    hr_locations.extend([DATASET_DIR / name, DATA_DIR / name])
HR_CSV_PATH = next((p for p in hr_locations if p.exists()), None)
assert HR_CSV_PATH is not None, (
    f"Could not find HR CSV. Looked for {HR_CSV_NAMES} in {DATASET_DIR} and {DATA_DIR}"
)
print(f"HR data file : {HR_CSV_PATH}")


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="white")

hr = pd.read_csv(DATA_DIR / "hr_with_features.csv")
hr["AttritionFlag"] = (hr["Attrition"] == "Yes").astype(int)
print(f"Loaded {len(hr):,} employees")


## Heatmap 1 — Tenure × Job Level attrition matrix

In [ ]:
tenure_bins = [-0.5, 1, 2, 3, 5, 7, 10, 15, 50]
tenure_labels = ["0-1", "1-2", "2-3", "3-5", "5-7", "7-10", "10-15", "15+"]
hr["tenure_bucket"] = pd.cut(hr["YearsAtCompany"], bins=tenure_bins, labels=tenure_labels)

heat = (hr.groupby(["tenure_bucket", "JobLevel"], observed=True)["AttritionFlag"]
        .mean().unstack() * 100)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(heat, annot=True, fmt=".1f", cmap="RdYlGn_r",
            cbar_kws={"label": "Attrition rate (%)"},
            linewidths=0.6, linecolor="white", ax=ax)
ax.set_title("Attrition Rate (%) — Tenure × Job Level\nDarker red = higher attrition risk",
             fontweight="bold", pad=12)
ax.set_xlabel("Job Level (1=junior, 5=executive)")
ax.set_ylabel("Tenure bucket (years)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "attrition_tenure_joblevel.png", dpi=160, bbox_inches="tight")
plt.show()


## Heatmap 2 — Department × OverTime

In [ ]:
heat2 = (hr.groupby(["Department", "OverTime"])["AttritionFlag"].mean() * 100).unstack()

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(heat2, annot=True, fmt=".1f", cmap="RdYlGn_r",
            cbar_kws={"label": "Attrition %"}, linewidths=0.6, linecolor="white", ax=ax)
ax.set_title("Attrition Rate (%) — Department × OverTime", fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "attrition_dept_overtime.png", dpi=160, bbox_inches="tight")
plt.show()


## Job Role attrition ranking

In [ ]:
role = (hr.groupby("JobRole")
        .agg(headcount=("EmployeeNumber", "count"),
             leavers=("AttritionFlag", "sum"),
             attrition_rate=("AttritionFlag", "mean"))
        .sort_values("attrition_rate", ascending=False))
role["attrition_rate_pct"] = (role["attrition_rate"] * 100).round(1)

fig, ax = plt.subplots(figsize=(11, 5.5))
overall = hr["AttritionFlag"].mean() * 100
colors = ["#C00000" if v > overall else "#5B9BD5" for v in role["attrition_rate_pct"]]
bars = ax.barh(role.index[::-1], role["attrition_rate_pct"][::-1], color=colors[::-1])
ax.axvline(overall, color="black", linestyle="--", linewidth=1.2,
           label=f"Company avg ({overall:.1f}%)")
ax.set_xlabel("Attrition rate (%)")
ax.set_title("Attrition by Job Role", fontweight="bold", pad=12)
ax.bar_label(bars, fmt="%.1f%%", padding=3)
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "attrition_by_jobrole.png", dpi=160, bbox_inches="tight")
plt.show()

role.to_csv(OUTPUTS_DIR / "attrition_by_jobrole.csv")


## Distance-from-home effect

In [ ]:
dist_bins = [-1, 5, 10, 20, 100]
dist_labels = ["<5 km", "5-10 km", "10-20 km", "20+ km"]
hr["distance_bucket"] = pd.cut(hr["DistanceFromHome"], bins=dist_bins, labels=dist_labels)

dist = (hr.groupby("distance_bucket", observed=True)
        .agg(headcount=("EmployeeNumber", "count"),
             attrition_rate=("AttritionFlag", "mean")))
dist["attrition_rate_pct"] = (dist["attrition_rate"] * 100).round(1)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(dist.index.astype(str), dist["attrition_rate_pct"], color="#5B9BD5")
ax.axhline(overall, color="black", linestyle="--", label=f"Company avg ({overall:.1f}%)")
ax.set_ylabel("Attrition rate (%)")
ax.set_xlabel("Distance from home")
ax.set_title("Attrition by Commute Distance", fontweight="bold", pad=12)
ax.bar_label(bars, fmt="%.1f%%", padding=3)
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "attrition_by_distance.png", dpi=160, bbox_inches="tight")
plt.show()


## Satisfaction scores vs attrition

In [ ]:
sat_cols = ["JobSatisfaction", "EnvironmentSatisfaction", "WorkLifeBalance", "JobInvolvement"]

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5), sharey=True)
for ax, col in zip(axes, sat_cols):
    s = hr.groupby(col)["AttritionFlag"].mean() * 100
    bars = ax.bar(s.index.astype(str), s.values, color="#5B9BD5")
    ax.axhline(overall, color="black", linestyle="--")
    ax.set_title(col, fontweight="bold")
    ax.set_xlabel("Score (1=low, 4=high)")
    ax.bar_label(bars, fmt="%.1f%%", padding=3, fontsize=9)

axes[0].set_ylabel("Attrition rate (%)")
plt.suptitle("Attrition vs Satisfaction Dimensions", fontweight="bold", y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "attrition_by_satisfaction.png", dpi=160, bbox_inches="tight")
plt.show()


✅ **Notebook 03 complete.** Move to `04_market_modeling_xlsx.ipynb`.